## LSTM (Long Short Term Memory):- 
Long Short term memory is a type of the sequential model, that takes one timestep at a time in order process it and produced two state, cell state and the hidden state.

Cell staate c(t) is the long term memory, that contains the data which are important for long term. It contains the understanding that the model has understood so far that is important in future timestep and they might depend upon but we dont know which and what will be important.

Hidden state h(t) is the short term memory, that contains the contextual understanding/summary or the snapshot of the lstm model throughout the process till the previous timestep. It does hold the information that is important and crucial for now.

It makes the LSTM, different from the RNN because, RNN stores everything in one state due to which the information that we will be loosing is the long term information, which removes the long term dependency/quality information and it causes the lack of prediciton quality.

Whereas, in LSTM there are two state, cell state only holds the information that are found in early timestep as well as the information that can be crucial for future timesteps. And the short term memory s like the hidden state of rnn, as it holds the contextual understanding till the previous timestep. And the short term memory contains the information crucial in current scenario. And this provides us the ability to play with the LTM and STM like, we can decide what should we forget or is no longer important, what should we consider the information that can be important in future and what are the information we neet to add to STM from LTM which can be crucial for next time step.

Provides us control to what to forget, what to remember and what to consider right now.

In [229]:
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter

In [230]:
nltk.download("punkt_tab")
nltk.download("punkt")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Nitro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Nitro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [231]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

### we cannot load all the text from the text.txt file all at once because of the RAM gettig full, so we will use generator to produce desired number of text at a time

In [232]:
def file():
    with open("smaller.txt", "r") as f:
            for line in f:
                yield line

In [233]:
def tokens_generator(f):
    tokens = []
    try:
        while True:
            text = next(f)
            tokens.extend(word_tokenize(text.lower()))
    except StopIteration:
        return tokens

In [234]:
def voccabulary():
    f = file()
    tokens = tokens_generator(f)
    voccab = {
        '<UNK>' : 0,
        '<PAD>' : 1,
    }
    for token in Counter(tokens):
        voccab[token] = len(voccab)
    return voccab

In [235]:
voccab = voccabulary()

In [236]:
voccab

{'<UNK>': 0,
 '<PAD>': 1,
 'usually': 2,
 'he': 3,
 'would': 4,
 'be': 5,
 'tearing': 6,
 'around': 7,
 'the': 8,
 'living': 9,
 'room': 10,
 'playing': 11,
 'with': 12,
 'his': 13,
 'toys': 14,
 'but': 15,
 'just': 16,
 'one': 17,
 'look': 18,
 'at': 19,
 'a': 20,
 'minion': 21,
 'sent': 22,
 'him': 23,
 'practically': 24,
 'catatonic': 25,
 'that': 26,
 'had': 27,
 'been': 28,
 'megan': 29,
 's': 30,
 'plan': 31,
 'when': 32,
 'she': 33,
 'got': 34,
 'dressed': 35,
 'earlier': 36,
 'd': 37,
 'seen': 38,
 'movie': 39,
 'almost': 40,
 'by': 41,
 'mistake': 42,
 'considering': 43,
 'was': 44,
 'little': 45,
 'young': 46,
 'for': 47,
 'pg': 48,
 'cartoon': 49,
 'older': 50,
 'cousins': 51,
 'along': 52,
 'her': 53,
 'brothers': 54,
 'mason': 55,
 'often': 56,
 'exposed': 57,
 'to': 58,
 'things': 59,
 'were': 60,
 'liked': 61,
 'think': 62,
 'being': 63,
 'surrounded': 64,
 'adults': 65,
 'and': 66,
 'kids': 67,
 'reason': 68,
 'why': 69,
 'such': 70,
 'good': 71,
 'talker': 72,
 'age': 

In [237]:
len(voccab)

10473

In [238]:
def text_to_indices(voccab):
    f = file()
    dataset = []
    max_len = 0
    # extract text from file
    try:
        while True:
            # tokenize
            indices = []
            text = next(f)
            tokens = word_tokenize(text.lower())
            if max_len < len(tokens):
                max_len = len(tokens)
            # replace with respective integer from voccab
            for token in tokens:
                indices.append(voccab[token])
            # preparing the dataset from indices
            for element in range(len(indices) - 1):
                dataset.append(indices[ : element + 2])
    except StopIteration:
        return  max_len, dataset

In [239]:
indices = text_to_indices(voccab)

In [240]:
indices

(72,
 [[2, 3],
  [2, 3, 4],
  [2, 3, 4, 5],
  [2, 3, 4, 5, 6],
  [2, 3, 4, 5, 6, 7],
  [2, 3, 4, 5, 6, 7, 8],
  [2, 3, 4, 5, 6, 7, 8, 9],
  [2, 3, 4, 5, 6, 7, 8, 9, 10],
  [2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
  [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
  [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],
  [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
  [15, 16],
  [15, 16, 17],
  [15, 16, 17, 18],
  [15, 16, 17, 18, 19],
  [15, 16, 17, 18, 19, 20],
  [15, 16, 17, 18, 19, 20, 21],
  [15, 16, 17, 18, 19, 20, 21, 22],
  [15, 16, 17, 18, 19, 20, 21, 22, 23],
  [15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
  [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25],
  [26, 27],
  [26, 27, 28],
  [26, 27, 28, 29],
  [26, 27, 28, 29, 30],
  [26, 27, 28, 29, 30, 31],
  [26, 27, 28, 29, 30, 31, 32],
  [26, 27, 28, 29, 30, 31, 32, 33],
  [26, 27, 28, 29, 30, 31, 32, 33, 34],
  [26, 27, 28, 29, 30, 31, 32, 33, 34, 23],
  [26, 27, 28, 29, 30, 31, 32, 33, 34, 23, 35],
  [26, 27, 28, 29, 30, 31, 32, 33, 34, 23, 35, 36],
  [3, 37],

### we need to create the dataset (inputs) for the LSTM model
As we are trying to make the next word predction so, in a sequence there can be different size of sentence so we need to perfrom the padding to make them of the same size as well.

In [241]:
def dataset_preparatioin(max_length, indices):
    dataset = []
    for i in indices:
        dataset.append([1]*(max_length - len(i)) + i)
    return dataset

In [242]:
dataset = dataset_preparatioin(indices[0], indices[1])

### Now, lets create the dataset and data loader for input and expected output from the model

In [243]:
class CustomDataSet(Dataset):
    def __init__(self, texts):
        self.features = torch.tensor(texts)[:, :-1].to(device)
        self.target = torch.tensor(texts)[:, -1].to(device)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return (self.features[index], self.target[index])

In [244]:
data = CustomDataSet(dataset)

In [245]:
data.__len__()

188210

In [246]:
data.__getitem__(0)

(tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2],
        device='cuda:0'),
 tensor(3, device='cuda:0'))

In [247]:
dataloader = DataLoader(data, batch_size = 64, shuffle = False)

In [248]:
for i,j in dataloader:
    print(i,j)
    break

tensor([[ 1,  1,  1,  ...,  1,  1,  2],
        [ 1,  1,  1,  ...,  1,  2,  3],
        [ 1,  1,  1,  ...,  2,  3,  4],
        ...,
        [ 1,  1,  1,  ..., 55, 44, 56],
        [ 1,  1,  1,  ..., 44, 56, 57],
        [ 1,  1,  1,  ..., 56, 57, 58]], device='cuda:0') tensor([ 3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21,
        22, 23, 24, 25, 27, 28, 29, 30, 31, 32, 33, 34, 23, 35, 36, 37, 38,  8,
        39, 40, 41, 42, 43,  3, 44, 20, 45, 46, 47,  8, 48, 49, 15, 12, 50, 51,
        52, 12, 53, 54, 55, 44, 56, 57, 58, 59], device='cuda:0')


In [249]:
class NextWordPredictor(nn.Module):

    def __init__(self, voccab):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings = len(voccab), embedding_dim = 100)
        self.lstm = nn.LSTM(input_size = 100, hidden_size = 164, num_layers = 4, batch_first = True)
        self.linear = nn.Linear(164, len(voccab))
    
    def forward(self, features):
        embed = self.embedding(features)
        lstm = self.lstm(embed)[1][0][-1]
        linear = self.linear(lstm)
        return linear

In [250]:
model = NextWordPredictor(voccab)
model.to(device=device)

NextWordPredictor(
  (embedding): Embedding(10473, 100)
  (lstm): LSTM(100, 164, num_layers=4, batch_first=True)
  (linear): Linear(in_features=164, out_features=10473, bias=True)
)

In [251]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)
epochs = 35

In [252]:
for epoch in range(epochs):
    net_loss = 0
    for features, target in dataloader:
        output = model(features)
        loss = loss_fn(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        net_loss = net_loss + loss.item()
    average_loss = net_loss/len(dataloader)
    print("epoch ===>", epoch, "loss ===>", average_loss)

epoch ===> 0 loss ===> 7.054389686091584
epoch ===> 1 loss ===> 6.820206301506585
epoch ===> 2 loss ===> 6.690133460076315
epoch ===> 3 loss ===> 6.6877372069490155
epoch ===> 4 loss ===> 6.687720325032856
epoch ===> 5 loss ===> 6.687719709084416
epoch ===> 6 loss ===> 6.687719642285058
epoch ===> 7 loss ===> 6.687719652661658
epoch ===> 8 loss ===> 6.6877196607683755
epoch ===> 9 loss ===> 6.687719664659601
epoch ===> 10 loss ===> 6.687719650553912
epoch ===> 11 loss ===> 6.68771965347233
epoch ===> 12 loss ===> 6.687719637583162
epoch ===> 13 loss ===> 6.68771964196079
epoch ===> 14 loss ===> 6.6877196704964375
epoch ===> 15 loss ===> 6.68771964828403
epoch ===> 16 loss ===> 6.687719662227585
epoch ===> 17 loss ===> 6.687719658822764
epoch ===> 18 loss ===> 6.687719643095731
epoch ===> 19 loss ===> 6.687719648770433
epoch ===> 20 loss ===> 6.687719647959762
epoch ===> 21 loss ===> 6.6877196304492506
epoch ===> 22 loss ===> 6.687719641474387
epoch ===> 23 loss ===> 6.6877196417986555


KeyboardInterrupt: 